In [1]:
# Source - https://stackoverflow.com/a
# Posted by G M, modified by community. See post 'Timeline' for change history
if 'google.colab' in str(get_ipython()):
  !git clone https://github.com/Vladislavicious/jenga_ml.git
  %cd jenga_ml
  !git switch dev

  !pip install -r requirements.txt
else:
  print('Not running on CoLab')

Not running on CoLab


In [2]:
import random
from environment import make_jenga_env
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env


In [3]:
n_blocks = 6
random.seed(123)
np.random.seed(123)
CHECK_STEPS = 1024

def make_env():
    return make_jenga_env(n_blocks=n_blocks, render=True)


In [4]:
num_envs = 8
env = make_vec_env(make_env, n_envs=num_envs, vec_env_cls=SubprocVecEnv)


In [5]:

model = PPO(
    "MlpPolicy",
    env,
    n_steps=CHECK_STEPS,
    batch_size=128,
    verbose=1,
    seed=123,

)

model.learn(total_timesteps=10_000)

Using cpu device
-----------------------------
| time/              |      |
|    fps             | 2324 |
|    iterations      | 1    |
|    time_elapsed    | 3    |
|    total_timesteps | 8192 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1497        |
|    iterations           | 2           |
|    time_elapsed         | 10          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.010829947 |
|    clip_fraction        | 0.111       |
|    clip_range           | 0.2         |
|    entropy_loss         | -6.18       |
|    explained_variance   | -0.0419     |
|    learning_rate        | 0.0003      |
|    loss                 | 0.381       |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00974    |
|    value_loss           | 23.9        |
-----------------------------------------


In [6]:
model.save("jenga_ppo_multithread")


In [7]:
single_env = make_env()

In [12]:
obs, _ = single_env.reset()
for _ in range(CHECK_STEPS):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = single_env.step(action)
    single_env.render()
    if terminated or truncated:
        obs, _ = single_env.reset()
    single_env.env.debug_output()

block 0: [-0.39952321 -0.23983145  0.00718846]
block 1: [ 0.35959672 -0.10557082  0.00737709]
block 2: [-0.1972078  -0.0035685   0.00730627]
block 3: [0.35321181 0.1447743  0.00734533]
block 4: [-0.06026627 -0.12204275  0.00734924]
block 5: [ 0.06237464 -0.27505456  0.00736258]
block 0: [-0.39954442 -0.23981699  0.00724096]
block 1: [ 0.35959582 -0.10556894  0.00738077]
block 2: [-0.19720613 -0.00356848  0.0073274 ]
block 3: [0.35321017 0.14476827 0.00735685]
block 4: [-0.06027164 -0.12203973  0.0073598 ]
block 5: [ 0.06237464 -0.28005456  0.00736258]
block 0: [-0.39955989 -0.23980644  0.00727925]
block 1: [ 0.35959514 -0.10556752  0.00738356]
block 2: [-0.19720487 -0.00356845  0.00734328]
block 3: [0.35320893 0.14476373 0.00736551]
block 4: [-0.06027568 -0.12203745  0.00736774]
block 5: [ 0.06737464 -0.28005456  0.00736258]
block 0: [-0.39957132 -0.23979866  0.0073075 ]
block 1: [ 0.35959463 -0.10556644  0.00738567]
block 2: [-0.19720393 -0.00356844  0.00735524]
block 3: [0.353208   0

In [9]:
model.save("model_100k.mod")

In [10]:
env.env.debug_output()

AttributeError: 'SubprocVecEnv' object has no attribute 'env'